In [4]:
# Generation de dataset
import numpy as np
import pandas as pd
import copy
import time

SIZE = 5

class HexMinimax:
    def __init__(self, size=SIZE):
        self.size = size

    def get_neighbors(self, r, c):
        neighbors = []
        for dr, dc in [(-1,0), (-1,1), (0,-1), (0,1), (1,-1), (1,0)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < self.size and 0 <= nc < self.size:
                neighbors.append((nr, nc))
        return neighbors

    def check_winner(self, board, player):
        visited = set()
        if player == 1: # Bleu (Haut -> Bas)
            starts = [(0, c) for c in range(self.size) if board[0, c] == 1]
            targets = set((self.size-1, c) for c in range(self.size))
        else: # Rouge (Gauche -> Droite)
            starts = [(r, 0) for r in range(self.size) if board[r, 0] == 2]
            targets = set((r, self.size-1) for r in range(self.size))
        
        stack = starts
        while stack:
            curr = stack.pop()
            if curr in targets: return True
            if curr not in visited:
                visited.add(curr)
                for n in self.get_neighbors(*curr):
                    if board[n] == player:
                        stack.append(n)
        return False

    def evaluate(self, board):
        if self.check_winner(board, 1): return 100
        if self.check_winner(board, 2): return -100
        return 0

    def minimax(self, board, depth, alpha, beta, maximizing_player):
        res = self.evaluate(board)
        if depth == 0 or res != 0:
            return res

        empty = list(zip(*np.where(board == 0)))
        if not empty: return 0

        if maximizing_player:
            max_eval = -float('inf')
            for r, c in empty:
                board[r, c] = 1
                eval = self.minimax(board, depth-1, alpha, beta, False)
                board[r, c] = 0
                max_eval = max(max_eval, eval)
                alpha = max(alpha, eval)
                if beta <= alpha: break
            return max_eval
        else:
            min_eval = float('inf')
            for r, c in empty:
                board[r, c] = 2
                eval = self.minimax(board, depth-1, alpha, beta, True)
                board[r, c] = 0
                min_eval = min(min_eval, eval)
                beta = min(beta, eval)
                if beta <= alpha: break
            return min_eval

    def get_best_move(self, board, player, depth=6):
        empty = list(zip(*np.where(board == 0)))
        best_score = -float('inf') if player == 1 else float('inf')
        best_move = empty[0]

        for r, c in empty:
            board[r, c] = player
            score = self.minimax(board, depth-1, -float('inf'), float('inf'), player == 2)
            board[r, c] = 0
            
            if player == 1 and score > best_score:
                best_score, best_move = score, (r, c)
            elif player == 2 and score < best_score:
                best_score, best_move = score, (r, c)
        
        return best_move

# --- BOUCLE DE GÉNÉRATION ---
dataset = []
engine = HexMinimax()

print("Début de la génération du dataset expert...")

for game in range(1500): 
    board = np.zeros((SIZE, SIZE), dtype=int)
    current_player = 1
    
    while True:
        # 1. Trouver le meilleur coup
        move = engine.get_best_move(board, current_player, depth=3)
        
        # 2. Enregistrer l'état AVANT le coup (X) et le coup choisi (y)
        # On encode le plateau en 5x5
        dataset.append({
            'board': board.flatten().tolist(),
            'move': move[0] * SIZE + move[1] # Index de 0 à 24
        })
        
        # 3. Jouer le coup
        board[move] = current_player
        
        if engine.check_winner(board, current_player) or not np.any(board == 0):
            break
        current_player = 3 - current_player # Alterne entre 1 et 2

    if game % 10 == 0:
        print(f"Partie {game} terminée...")

df = pd.DataFrame(dataset)
df.to_csv("hex_expert_data.csv", index=False)
print("Dataset sauvegardé dans hex_expert_data.csv")

🚀 Début de la génération du dataset expert...
Partie 0 terminée...
Partie 10 terminée...
Partie 20 terminée...
Partie 30 terminée...
Partie 40 terminée...
Partie 50 terminée...
Partie 60 terminée...
Partie 70 terminée...
Partie 80 terminée...
Partie 90 terminée...
Partie 100 terminée...
Partie 110 terminée...
Partie 120 terminée...
Partie 130 terminée...
Partie 140 terminée...
Partie 150 terminée...
Partie 160 terminée...
Partie 170 terminée...
Partie 180 terminée...
Partie 190 terminée...
Partie 200 terminée...
Partie 210 terminée...
Partie 220 terminée...
Partie 230 terminée...
Partie 240 terminée...
Partie 250 terminée...
Partie 260 terminée...
Partie 270 terminée...
Partie 280 terminée...
Partie 290 terminée...
Partie 300 terminée...
Partie 310 terminée...
Partie 320 terminée...
Partie 330 terminée...
Partie 340 terminée...
Partie 350 terminée...
Partie 360 terminée...
Partie 370 terminée...
Partie 380 terminée...
Partie 390 terminée...
Partie 400 terminée...
Partie 410 terminée...

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

# 1. Préparation des données
df = pd.read_csv("hex_expert_data.csv")
X, y = [], []

for _, row in df.iterrows():
    board = np.array(eval(row['board'])).reshape(5, 5)
    # On crée les 2 canaux (Joueur 1 / Joueur 2)
    layer1 = (board == 1).astype(float)
    layer2 = (board == 2).astype(float)
    X.append(np.stack([layer1, layer2], axis=0)) # PyTorch préfère (Canal, H, W)
    y.append(row['move'])

X = torch.tensor(np.array(X), dtype=torch.float32)
y = torch.tensor(np.array(y), dtype=torch.long)

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

# 2. Architecture du CNN (Équivalent à ce qu'on a fait en TF)
class HexCNN(nn.Module):
    def __init__(self):
        super(HexCNN, self).__init__()
        self.conv1 = nn.Conv2d(2, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.fc1 = nn.Linear(128 * 5 * 5, 256)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(256, 25)

    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        x = torch.relu(self.bn2(self.conv2(x)))
        x = x.view(-1, 128 * 5 * 5)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

# 3. Entraînement rapide
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HexCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print("🚀 Entraînement PyTorch en cours...")
for epoch in range(50):
    for boards, moves in loader:
        boards, moves = boards.to(device), moves.to(device)
        optimizer.zero_grad()
        outputs = model(boards)
        loss = criterion(outputs, moves)
        loss.backward()
        optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/50 | Loss: {loss.item():.4f}")

# 4. Sauvegarde
torch.save(model.state_dict(), "hex_model_pytorch.pth")
print("✅ Modèle sauvegardé : hex_model_pytorch.pth")

ModuleNotFoundError: No module named 'torch'